In [ ]:
from tqdm.auto import tqdm
from pdf2image import convert_from_path

# Path to your PDF
pdf_path = "../data/KR Notation Guide_2025.pdf"
filename = pdf_path.split("/")[-1].replace(".pdf", "")

# Convert all pages to images
images = convert_from_path(pdf_path, dpi=300)

# Save each page as a separate image
for i, page in tqdm(enumerate(images)):
    image_path = f"../data/notation/{filename}_{i}.png"
    page.save(image_path, "PNG")
    print(f"Saved: {image_path}")


Saved: page_1.png


In [ ]:
import os
import pickle
from time import time

import pandas as pd
from langchain_core.documents import Document

from docling.document_converter import DocumentConverter, ImageFormatOption
from docling.models.tesseract_ocr_cli_model import TesseractCliOcrOptions



file_path = "../data/page_1.png"
lv1_cat, lv2_cat = "Rule", "KR"

path = file_path.replace("\\", "/")
filename = path.split("/")[-1]
first_sentence = f"This page explains {filename.replace(".pdf", "")} that belongs to {lv1_cat} and  {lv2_cat} categories.\n"


# Configure OCR for image input
image_options = ImageFormatOption(
    ocr_options=TesseractCliOcrOptions(force_full_page_ocr=True),
    do_table_structure=True,
    table_structure_options={"do_cell_matching": True},
)

converter = DocumentConverter(
    format_options={"image": image_options}
)

start_time = time()
conv_res = converter.convert(file_path).document

tables = []
# Print all tables as Markdown
for table_ix, table in enumerate(conv_res.tables):
    table_df: pd.DataFrame = table.export_to_dataframe(doc=conv_res)
    page_num = table.prov[0].page_no if table.prov else "Unknown"
    extracted_table = first_sentence + table_df.to_markdown()
    lang_table = Document(page_content=extracted_table, metadata={'filename': filename, 'lv1_cat': lv1_cat, 'lv2_cat': lv2_cat, 'page':str(page_num)})
    tables.append(lang_table)
    
parsed_foldername = f"{lv1_cat}_{lv2_cat}"
if not os.path.exists(f"../docs/{parsed_foldername}_img"):
    os.makedirs(f"../docs/{parsed_foldername}_img")

parsed_filename = filename.replace(".png", "").replace(".jpg", "")
with open(f"../docs/{parsed_foldername}_img/{parsed_filename}.pkl", 'ab') as file:
    pickle.dump(tables, file)
        

end_time = time() - start_time
print(f"Document converted and tables exported in {end_time:.2f} seconds.")

2025-10-08 09:10:32,648 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]


2025-10-08 09:10:34,703 - INFO - Going to convert document batch...
2025-10-08 09:10:34,704 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e647edf348883bed75367b22fbe60347
2025-10-08 09:10:34,768 - INFO - Loading plugin 'docling_defaults'
2025-10-08 09:10:34,773 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-10-08 09:10:34,828 - INFO - Loading plugin 'docling_defaults'
2025-10-08 09:10:34,834 - INFO - Registered ocr engines: ['easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-10-08 09:10:35,118 - INFO - Accelerator device: 'cpu'
2025-10-08 09:10:38,162 - INFO - Accelerator device: 'cpu'
2025-10-08 09:10:40,321 - INFO - Accelerator device: 'cpu'
2025-10-08 09:10:40,875 - INFO - Processing document page_1.png
d:\my_parser\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
2025-10-08

Document converted and tables exported in 109.34 seconds.


In [5]:
import pickle

# Open the file containing the pickled data in binary read mode ('rb')
with open('../docs/Rule_KR_img/page_1.png.pkl', 'rb') as file:
    # Load the pickled object from the file
    loaded_object = pickle.load(file)

In [6]:
loaded_object

[Document(metadata={'filename': 'page_1.png', 'lv1_cat': 'Rule', 'lv2_cat': 'KR', 'page': '1'}, page_content='This page explains page_1.png that belongs to Rule and  KR categories.\n|    | Location                                                   | Up to 10,000 GT.   | Up to 10,000 GT.N2   | Up to 10,000 GT.N3   | 210,000 GT.N1   | 210,000 GT.N2   | 210,000 GT.N3   |\n|---:|:-----------------------------------------------------------|:-------------------|:---------------------|:---------------------|:----------------|:----------------|:----------------|\n|  0 | Navigation spaces and control stations                     |                    |                      |                      |                 |                 |                 |\n|  1 | Radio rooms                                                | 60                 | 58                   | 55                   | 60              | 58              | 55              |\n|  2 | Navigating bridges, chartroom; radar rooms         

In [7]:
from IPython.display import Markdown
Markdown(loaded_object[0].page_content)

This page explains page_1.png that belongs to Rule and  KR categories.
|    | Location                                                   | Up to 10,000 GT.   | Up to 10,000 GT.N2   | Up to 10,000 GT.N3   | 210,000 GT.N1   | 210,000 GT.N2   | 210,000 GT.N3   |
|---:|:-----------------------------------------------------------|:-------------------|:---------------------|:---------------------|:----------------|:----------------|:----------------|
|  0 | Navigation spaces and control stations                     |                    |                      |                      |                 |                 |                 |
|  1 | Radio rooms                                                | 60                 | 58                   | 55                   | 60              | 58              | 55              |
|  2 | Navigating bridges, chartroom; radar rooms                 | 65                 | 63                   | 60                   | 65              | 63              | 60              |
|  3 | Look-out post, incl. navigating   bridge wings and windows | 70                 | 70                   | 70                   | 70              | 70              | 70              |
|  4 | Control stations                                           | 65                 | 63                   | 60                   | 65              | 63              | 60              |
|  5 | Accommodations Spaces                                      |                    |                      |                      |                 |                 |                 |
|  6 | Cabin                                                      | 60                 | 55                   | 50                   | 55              | 53              | 50              |
|  7 | hospitals                                                  | 60                 | 58                   | 55                   | 55              | 53              | 50              |
|  8 | Messroom                                                   | 65                 | 60                   | 55                   | 60              | 58              | 55              |
|  9 |                                                            | 65                 | 63                   | 60                   | 60              | 58              | 55              |
| 10 | Open deck recreation area                                  | 75                 | 73                   | 70                   | 75              | 73              | 70              |
| 11 | Staircase and passages in accommodation                    | 75                 | 73                   | 70                   | 75              | 73              | 70              |
| 12 | Service spaces                                             |                    |                      |                      |                 |                 |                 |
| 13 | Galley                                                     | 75                 | 73                   | 70                   | 75              | 73              | 70              |
| 14 | Other services                                             | 75                 | 73                   | 70                   | 75              | 73              | 70              |
| 15 | Work spaces                                                |                    |                      |                      |                 |                 |                 |
| 16 | Machinery spaces                                           | 110                | 110                  | 110                  | 110             | 110             | 110             |
| 17 | Machinery control rooms                                    | 75                 | 73                   | 70                   | 75              | 73              | 70              |
| 18 | Other work spaces                                          | 85                 | 83                   | 80                   | 85              | 83              | 80              |
| 19 | Normally unoccupied spaces                                 |                    |                      |                      |                 |                 |                 |
| 20 | Spaces   referred in 301. 7                                | 90                 | 90                   | 90                   | 90              | 90              | 90              |